# Kaggle Inline MNIST 训练模板

使用与仓库示例一致的 `mnist.npz` 数据格式、MLP 结构和恢复语义。
训练实验代码直接写在 Notebook 中，并在启动前落盘为可导入模块。

运行前挂载 MNIST NPZ Dataset，并在 Kaggle Secrets 中配置：
`ALIST_USER`、`ALIST_PWD`、`WECOM_CORP_ID`、`WECOM_CORP_SECRET`、
`WECOM_AGENT_ID`。


In [ ]:
from pathlib import Path

# 替换为你的 dl_helper 仓库和固定版本。
DL_HELPER_REPO_URL = "https://github.com/your-account/dl_helper.git"
DL_HELPER_REF = "master"

# 与旧 MNIST NPZ 示例相同的数据集格式；也可替换为其他等价 mnist.npz 路径。
DATA_PATH = "/kaggle/input/datasets/vikramtiwari/mnist-numpy/mnist.npz"
PROJECT_DIR = Path("/kaggle/working/inline-mnist")
EXPERIMENT_PATH = PROJECT_DIR / "my_experiment.py"
CONFIG_PATH = PROJECT_DIR / "configs" / "kaggle.yaml"
EXPERIMENT = "my_experiment:build_experiment"
RUN_ID = "inline-mnist-15epoch"
SOURCE_REVISION = "inline-mnist-v1"

ALIST_HOST = "http://139.196.47.52"
ALIST_BASE_PATH = "/dl-helper/inline-mnist"
WECOM_TO_USER = "@all"

for name, value in (("DL_HELPER_REPO_URL", DL_HELPER_REPO_URL),
                    ("DL_HELPER_REF", DL_HELPER_REF),
                    ("RUN_ID", RUN_ID),
                    ("SOURCE_REVISION", SOURCE_REVISION),
                    ("ALIST_BASE_PATH", ALIST_BASE_PATH)):
    if not value or any(character.isspace() for character in value):
        raise ValueError(f"{name} 必须是非空且不含空白的字符串")
    if "your-account" in value:
        raise ValueError(f"请先把 {name} 替换为实际值")
if not Path(DATA_PATH).is_file():
    raise FileNotFoundError(f"MNIST 数据不存在，请先挂载 Dataset: {DATA_PATH}")

print("data:", DATA_PATH)
print("run id:", RUN_ID)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def run_checked(argv, *, cwd=None):
    print("+", " ".join(argv))
    process = subprocess.run(argv, cwd=cwd, text=True, encoding="utf-8")
    if process.returncode != 0:
        raise RuntimeError(f"命令失败，退出码 {process.returncode}: {' '.join(argv)}")
    return process


def clone_fixed(repo_url, ref, target):
    if target.exists():
        raise RuntimeError(f"目标目录已存在，请新建 Kaggle Session 后重试: {target}")
    run_checked(["git", "clone", repo_url, str(target)])
    run_checked(["git", "checkout", ref], cwd=target)
    head = run_checked(["git", "rev-parse", "HEAD"], cwd=target).stdout.strip()
    expected = run_checked(
        ["git", "rev-parse", f"{ref}^{{commit}}"], cwd=target
    ).stdout.strip()
    if head.lower() != expected.lower():
        raise RuntimeError(f"checkout HEAD 不匹配: {head} != {expected}")
    return target


DL_HELPER_DIR = clone_fixed(
    DL_HELPER_REPO_URL,
    DL_HELPER_REF,
    Path("/kaggle/working/dl-helper"),
)
os.environ["DL_HELPER_GIT_REPO"] = DL_HELPER_REPO_URL
os.environ["DL_HELPER_REPO_DIR"] = str(DL_HELPER_DIR)
run_checked(
    [sys.executable, str(DL_HELPER_DIR / "envs" / "kaggle_bootstrap.py")],
    cwd=DL_HELPER_DIR,
)
PROJECT_DIR.mkdir(parents=True, exist_ok=False)
(PROJECT_DIR / "configs").mkdir()


In [ ]:
%%writefile /kaggle/working/inline-mnist/my_experiment.py
"""Notebook 内联定义的 MNIST 实验；语义与 examples/experiments/mnist.py 保持一致。"""
from __future__ import annotations

import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset

from dl_helper.training.contracts import (
    DataIdentity,
    ResumableMapDataModule,
    TorchExperiment,
)
from dl_helper.training.task import MulticlassClassificationTask


class MNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


def _load_npz(path: str) -> tuple[np.ndarray, np.ndarray]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"MNIST 数据路径不存在: {path!r}")
    with np.load(path, allow_pickle=False) as data:
        keys = set(data.files)
        if {"images", "labels"}.issubset(keys):
            images = data["images"]
            labels = data["labels"]
        elif {"x_train", "x_test", "y_train", "y_test"}.issubset(keys):
            images = np.concatenate((data["x_train"], data["x_test"]), axis=0)
            labels = np.concatenate((data["y_train"], data["y_test"]), axis=0)
        else:
            raise ValueError(
                "MNIST NPZ 必须包含 images/labels 或完整的 "
                "x_train/x_test/y_train/y_test 键；实际键为 "
                f"{sorted(keys)!r}"
            )
        if images.ndim != 3 or images.shape[1:] != (28, 28):
            raise ValueError(f"MNIST images 必须为 [N, 28, 28]，实际为 {images.shape!r}")
        if labels.ndim != 1 or labels.shape[0] != images.shape[0]:
            raise ValueError(
                f"MNIST labels 必须为与 images 等长的一维数组，实际为 {labels.shape!r}"
            )
        images = images.astype(np.float32) / 255.0
        labels = labels.astype(np.int64)
    return images, labels


def _collate(batch):
    images, labels = zip(*batch)
    return torch.stack(images), torch.stack(labels)


def _mnist_dm(config: dict):
    data_path = config["data_path"]
    if not isinstance(data_path, str) or not data_path:
        raise ValueError("mnist 需要显式 data_path")
    images, labels = _load_npz(data_path)
    n = images.shape[0]
    n_train = int(n * 0.8)
    train_ds = TensorDataset(
        torch.from_numpy(images[:n_train]), torch.from_numpy(labels[:n_train])
    )
    val_ds = TensorDataset(
        torch.from_numpy(images[n_train:]), torch.from_numpy(labels[n_train:])
    )
    return ResumableMapDataModule(
        DataIdentity("mnist", "1.0", f"fp-mnist-{os.path.getsize(data_path)}"),
        lambda: train_ds,
        _collate,
        batch_size=64,
        shuffle=False,
        val_dataset_factory=lambda: val_ds,
        val_batch_size=64,
    )


def build_experiment(config: dict) -> TorchExperiment:
    def task_factory():
        return MulticlassClassificationTask(num_classes=10)

    def optimizer_factory(params):
        return torch.optim.Adam(params, lr=float(config.get("lr", 0.001)))

    return TorchExperiment(
        name="mnist",
        backend="torch",
        model_factory=MNISTMLP,
        datamodule_factory=lambda: _mnist_dm(config),
        task_factory=task_factory,
        optimizer_factory=optimizer_factory,
        scheduler_factory=lambda optimizer: None,
        model_config=dict(config),
    )


In [ ]:
import yaml

config = {
    "schema_version": 1,
    "run": {
        "name": "inline-mnist",
        "id": RUN_ID,
        "output_root": None,
        "source_revision": SOURCE_REVISION,
        "seed": 42,
        "tags": {},
    },
    "experiment": {"data_path": DATA_PATH, "lr": 0.001},
    "training": {"max_epochs": 15, "log_every_steps": 20},
    "backend": {
        "type": "torch",
        "torch": {
            "gradient_accumulation_steps": 1,
            "mixed_precision": "no",
            "compile": False,
            "clip_grad_norm": 1.0,
            "deterministic": "strict",
            "matmul_precision": "high",
            "find_unused_parameters": False,
        },
        "sklearn": None,
    },
    "distributed": {"num_processes": 1},
    "selection": {
        "metric": "val/loss",
        "mode": "min",
        "patience": 30,
        "min_delta": 0.0,
    },
    "checkpoint": {
        "every_epochs": 5,
        "every_optimizer_steps": None,
        "keep_last": 2,
    },
    "report": {
        "enabled": True,
        "curve_sample_limit": 100000,
        "prediction_sample_limit": 10000,
        "prediction_splits": ["val"],
    },
    "remote": {
        "type": "alist",
        "host": ALIST_HOST,
        "base_path": ALIST_BASE_PATH,
        "user_secret_key": "ALIST_USER",
        "password_secret_key": "ALIST_PWD",
        "connect_timeout_seconds": 600,
        "read_timeout_seconds": 600,
        "max_attempts": 3,
        "async_upload": False,
        "failure_policy": "required",
    },
    "notifications": {
        "type": "wecom",
        "corp_id_secret_key": "WECOM_CORP_ID",
        "corp_secret_key": "WECOM_CORP_SECRET",
        "agent_id_secret_key": "WECOM_AGENT_ID",
        "to_user": WECOM_TO_USER,
        "connect_timeout_seconds": 10,
        "read_timeout_seconds": 30,
        "max_attempts": 3,
        "failure_policy": "required",
    },
}
CONFIG_PATH.write_text(
    yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8"
)
print("配置已写入:", CONFIG_PATH)


In [ ]:
train_argv = [
    sys.executable,
    "-m",
    "dl_helper.training.cli",
    "train",
    "--project-dir", str(PROJECT_DIR),
    "--config", str(CONFIG_PATH),
    "--experiment", EXPERIMENT,
    "--run-id", RUN_ID,
]
train_process = subprocess.run(train_argv, cwd=DL_HELPER_DIR, text=True, encoding="utf-8")
if train_process.returncode == 75:
    print("训练因 Kaggle 预算保护暂停；新 Session 重跑本单元格即可自动继续。")
elif train_process.returncode != 0:
    raise RuntimeError(f"训练失败，退出码: {train_process.returncode}")
else:
    print("训练完成。")
